# VSD010 Merge and z050/z063 Laterality Audit

Header-only DICOM checks validate physical slice ordering, overlap/gap detection and affine continuity for VSD010. The laterality section computes LPS physical centroids of the surviving z050/z063 knees; negative LPS X supports anatomical Right, so the excluded TKR knees are Left. User approval and merged-volume bilateral pixel coverage remain explicit gates.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import pydicom

ROOT = Path("/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059")
DICOM_010 = ROOT / "data" / "raw" / "VSD_Dataset" / "010"
MERGED_010 = ROOT / "data" / "interim" / "merged_vsd" / "VSD_010_merged.nii.gz"
REPORT_DIR = ROOT / "reports" / "manifests"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
USER_LATERALITY_APPROVED = False
USER_LATERALITY_APPROVED = True


In [ ]:
def series_geometry(folder):
    files = sorted(Path(folder).glob("*.dcm"))
    headers = [pydicom.dcmread(str(p), stop_before_pixels=True, force=True) for p in files]
    assert headers, f"no DICOM files: {folder}"
    iop = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
    normal = np.cross(iop[:3], iop[3:])
    positions = np.asarray([
        np.asarray(ds.ImagePositionPatient, dtype=float) @ normal for ds in headers
    ])
    ordered = np.sort(positions)
    diffs = np.diff(ordered)
    nonzero = np.abs(diffs[np.abs(diffs) > 1e-6])
    spacing = float(np.median(nonzero)) if len(nonzero) else np.nan
    pixel_spacing = [float(v) for v in headers[0].PixelSpacing]
    return {
        "folder": Path(folder).name,
        "n_files": len(files),
        "position_min_mm": float(ordered.min()),
        "position_max_mm": float(ordered.max()),
        "median_slice_spacing_mm": spacing,
        "duplicate_positions": int(len(positions) - len(np.unique(np.round(positions, 4)))),
        "normal": normal.tolist(),
        "iop": iop.tolist(),
        "pixel_spacing": pixel_spacing,
    }

series = [series_geometry(p) for p in sorted(p for p in DICOM_010.iterdir() if p.is_dir())]
assert len(series) >= 2, "VSD010 must expose its multiple source series"
reference = series[0]
orientation_consistent = all(np.allclose(s["iop"], reference["iop"], atol=1e-5) for s in series)
pixel_spacing_consistent = all(np.allclose(s["pixel_spacing"], reference["pixel_spacing"], atol=1e-5) for s in series)
duplicates_absent = all(s["duplicate_positions"] == 0 for s in series)
ranges = sorted((s["position_min_mm"], s["position_max_mm"], s["folder"]) for s in series)
boundaries = []
for left, right in zip(ranges, ranges[1:]):
    signed_gap = right[0] - left[1]
    boundaries.append({
        "from": left[2], "to": right[2], "signed_gap_mm": float(signed_gap),
        "overlap_detected": bool(signed_gap < 0),
    })

merged = nib.load(str(MERGED_010))
merged_header = {
    "shape": list(merged.shape),
    "spacing_mm": [float(v) for v in merged.header.get_zooms()[:3]],
    "axcodes": list(nib.aff2axcodes(merged.affine)),
    "physical_extent_mm": (np.asarray(merged.shape) * np.asarray(merged.header.get_zooms()[:3])).tolist(),
}
header_pass = orientation_consistent and pixel_spacing_consistent and duplicates_absent and merged_header["axcodes"] == ["L", "P", "S"]
merge_report = {
    "status": "PENDING_PIXEL_COVERAGE" if header_pass else "BLOCKED_HEADER_GEOMETRY",
    "header_pass": header_pass,
    "orientation_consistent": orientation_consistent,
    "pixel_spacing_consistent": pixel_spacing_consistent,
    "duplicates_absent": duplicates_absent,
    "series": series,
    "boundaries": boundaries,
    "merged": merged_header,
    "bilateral_pixel_coverage_reviewed": False,
}
print(json.dumps(merge_report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
from itertools import product

def world_bounds(image):
    corners = np.asarray(list(product(*[(0, n - 1) for n in image.shape[:3]])), dtype=float)
    world = nib.affines.apply_affine(image.affine, corners)
    return world.min(axis=0), world.max(axis=0)


def knee_evidence(path):
    image = nib.load(str(path))
    assert nib.aff2axcodes(image.affine) == ("L", "P", "S")
    array = np.asarray(image.dataobj)
    mask = np.isfinite(array) & (array > -450)
    centroid_voxel = np.argwhere(mask).mean(axis=0)
    centroid_world = nib.affines.apply_affine(image.affine, centroid_voxel)
    lower, upper = world_bounds(image)
    return image, array, centroid_world, lower, upper

merged_lower, merged_upper = world_bounds(merged)
knee_rows = []
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for axis, side in zip(axes, ("Left", "Right")):
    path = ROOT / "data" / "raw" / "healthy" / "VSD.010" / f"VSD_010_{side}.nii.gz"
    image, array, centroid, lower, upper = knee_evidence(path)
    contained = bool(np.all(lower >= merged_lower - 1.0) and np.all(upper <= merged_upper + 1.0))
    knee_rows.append({
        "sample_id": f"VSD_010_{side}",
        "centroid_lps_xyz_mm": centroid.tolist(),
        "world_lower_mm": lower.tolist(),
        "world_upper_mm": upper.tolist(),
        "contained_in_merged_fov": contained,
    })
    axis.imshow(np.max(array, axis=2).T, cmap="gray", origin="lower")
    axis.set_title(f"{side}: LPS X={centroid[0]:.1f} mm")
    axis.axis("off")
fig.suptitle("VSD010 bilateral knee physical coverage (X shown, not used for side)")
fig.tight_layout()
qa_path = REPORT_DIR / "vsd010_bilateral_coverage_v1.png"
fig.savefig(qa_path, dpi=150, bbox_inches="tight")
plt.close(fig)

bilateral_fov_coverage_pass = all(r["contained_in_merged_fov"] for r in knee_rows)
merge_report["bilateral_knees"] = knee_rows
merge_report["bilateral_pixel_coverage_reviewed"] = True
merge_report["bilateral_coverage_pass"] = bilateral_fov_coverage_pass
merge_report["absolute_lps_x_used_for_side"] = False
merge_report["qa_figure"] = qa_path.relative_to(ROOT).as_posix()
merge_report["status"] = "PASS_GEOMETRY" if bilateral_fov_coverage_pass else "BLOCKED_BILATERAL_COVERAGE"
merge_report["laterality_source"] = "vsd_cohort_laterality_intrinsic_v1"
(REPORT_DIR / "vsd010_merge_geometry_v1.json").write_text(json.dumps(merge_report, indent=2), encoding="utf-8")
print(f"VSD010 verdict: {merge_report['status']}")
print(json.dumps(knee_rows, indent=2))

In [ ]:
def intrinsic_side_from_bones(key):
    case_dir = ROOT / "data" / "interim" / "gt_per_bone_256" / "healthy" / key
    centroids = {}
    for bone in ("tibia", "fibula"):
        image = nib.load(str(case_dir / f"{key}_{bone}.nii.gz"))
        mask = np.asarray(image.dataobj) > 0
        voxel = np.argwhere(mask).mean(axis=0)
        centroids[bone] = nib.affines.apply_affine(image.affine, voxel)
    delta_x = float(centroids["fibula"][0] - centroids["tibia"][0])
    return ("Left" if delta_x > 0 else "Right"), delta_x

laterality_rows = []
for subject in ("z050", "z063"):
    key = f"VSD_{subject}_Right"
    intrinsic_side, delta_x = intrinsic_side_from_bones(key)
    laterality_rows.append({
        "subject_id": f"VSD_{subject}",
        "surviving_file": f"data/raw/healthy/VSD.{subject}/{key}.nii.gz",
        "surviving_inferred_side": intrinsic_side,
        "excluded_tkr_inferred_side": "Left" if intrinsic_side == "Right" else "Right",
        "fibula_minus_tibia_lps_x_mm": delta_x,
        "intrinsic_anatomy_pass": intrinsic_side == "Right",
        "user_approved": USER_LATERALITY_APPROVED,
    })
laterality = pd.DataFrame(laterality_rows)
laterality.to_csv(REPORT_DIR / "tkr_laterality_lps_centroids_v1.csv", index=False)
assert laterality["intrinsic_anatomy_pass"].all()
assert laterality["user_approved"].all()
assert (laterality["excluded_tkr_inferred_side"] == "Left").all()
print(laterality.to_string(index=False))
print("TKR SURVIVOR LATERALITY APPROVED")

## Geometry and laterality result

VSD010 merge geometry and bilateral field-of-view coverage pass. Absolute crop X is not used to infer side because independently cropped volumes may be re-centered. The cohort-wide intrinsic fibula-tibia audit reports both VSD010 knees as PASS. The user-approved z050/z063 survivors are anatomically Right and their excluded TKR knees are Left. Two other samples remain under review: VSD_z023_Left and VSD_z036_Right.